# Brain MRI FastAPI Backend — Project-Compatible Colab Notebook

This notebook creates and runs a FastAPI backend using the **actual project model and layout**:
- Google Drive root: `BrainMRI_Data`
- Registered segmentation model: `seg-model v1`
- Export created by `09_m4_t01_export_model.ipynb`: `models/seg-model-v1-torchscript`
- Modalities: `t1`, `t1ce`, `t2`, `flair` (cleaned/standardized NIfTI files)
- Regions and labels: `TC = {1,3}`, `WT = {1,2,3}`, `ET = {3}`

## API requirements covered
- `GET /health`
- `POST /predict`: one scan with four modality uploads; refuses unsupported file types and validates 3D geometry.
- `POST /predict/longitudinal`: two or more visits for the same patient; predicts each visit and calculates percentage volume changes between consecutive visits.
- JSON response includes segmentation mask path, tumor volumes, timings, radiomics status, and longitudinal changes when applicable.
- Includes a small API test for wrong file types.

**Important:** Upload cleaned/standardized scans matching the project's model input conventions. This is not a raw DICOM conversion or a clinical diagnostic tool. The notebook checks for the registered TorchScript export; if it is missing, it runs the repository's `scripts/export_model.py` using the verified model checkpoint. It does not retrain the model.


In [1]:
# 1. Mount Drive and configure the project paths
from pathlib import Path
import os, sys, json, subprocess, textwrap, time

from google.colab import drive
drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/BrainMRI_Data")
REPO = Path("/content/repo")
MODEL_DIR = BASE / "runs/M2-T04_nnunet/nnUNet_results/Dataset501_BraTS2021/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres"
EXPORT_DIR = BASE / "models/seg-model-v1-torchscript"
API_DIR = BASE / "api_backend"
OUT_DIR = BASE / "runs/API_Backend"
OUT_DIR.mkdir(parents=True, exist_ok=True)
API_DIR.mkdir(parents=True, exist_ok=True)

print("Drive project exists:", BASE.exists())
print("Model directory:", MODEL_DIR)
print("Export directory:", EXPORT_DIR)


Mounted at /content/drive
Drive project exists: True
Model directory: /content/drive/MyDrive/BrainMRI_Data/runs/M2-T04_nnunet/nnUNet_results/Dataset501_BraTS2021/nnUNetTrainer_50epochs__nnUNetPlans__3d_fullres
Export directory: /content/drive/MyDrive/BrainMRI_Data/models/seg-model-v1-torchscript


In [2]:
print("Data root exists:", BASE.exists())
print("Model directory exists:", MODEL_DIR.exists())
print("Fold 0 exists:", (MODEL_DIR / "fold_0").exists())
print("Export directory exists:", EXPORT_DIR.exists())

fold_dir = MODEL_DIR / "fold_0"

if fold_dir.exists():
    print("Model files:")
    for file in list(fold_dir.iterdir())[:10]:
        print("-", file.name)

Data root exists: True
Model directory exists: True
Fold 0 exists: True
Export directory exists: True
Model files:
- training_log_2026_10_9_20_44_18.txt
- training_log_2026_10_10_11_22_28.txt
- validation
- checkpoint_best.pth
- progress.png
- training_log_2026_10_10_12_45_34.txt
- debug.json
- training_log_2026_10_10_14_36_44.txt


In [3]:
# 2. Get the same repository branch used by the project's M4 export notebook
BRANCH = "main"
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "checkout", "-q", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "-q", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH,
                    "https://github.com/nhahub/NHA-5-187.git", str(REPO)], check=True)

sys.path.insert(0, str(REPO / "src"))
print("Repository ready:", REPO)
print(subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"], text=True).strip())


Repository ready: /content/repo
93c66f9


In [ ]:
# 3. Install core inference/API dependencies; PyRadiomics is handled separately as optional.
%pip -q install "nnunetv2==2.8.1" "fastapi>=0.115,<1.0" "uvicorn[standard]>=0.30,<1.0" python-multipart nibabel SimpleITK pandas requests


In [4]:
import importlib

modules = {
    "PyTorch": "torch",
    "nnU-Net": "nnunetv2",
    "FastAPI": "fastapi",
    "Uvicorn": "uvicorn",
    "File uploads": "multipart",
    "NIfTI reader": "nibabel",
    "SimpleITK": "SimpleITK",
    "Pandas": "pandas",
    "Requests": "requests",
}

failed = []

for name, module_name in modules.items():
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, "__version__", "installed")
        print(f"OK - {name}: {version}")
    except Exception as exc:
        failed.append(name)
        print(f"ERROR - {name}: {type(exc).__name__}: {exc}")

print("\nDEPENDENCIES_OK" if not failed else
      f"DEPENDENCIES_FAILED: {failed}")

OK - PyTorch: 2.11.0+cpu
ERROR - nnU-Net: ModuleNotFoundError: No module named 'nnunetv2'
OK - FastAPI: 0.141.1
OK - Uvicorn: 0.54.0
OK - File uploads: 0.0.32
OK - NIfTI reader: 5.4.2
ERROR - SimpleITK: ModuleNotFoundError: No module named 'SimpleITK'
OK - Pandas: 2.2.3
OK - Requests: 2.32.4
DEPENDENCIES_FAILED: ['nnU-Net', 'SimpleITK']


In [5]:
# 3a. Optional PyRadiomics install.
# PyPI source builds can fail on Colab's Python 3.13. Try the project's previously used GitHub source.
import subprocess, sys
rad_install = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "git+https://github.com/AIM-Harvard/pyradiomics.git"],
    text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT
)
print(rad_install.stdout[-5000:] if rad_install.stdout else "(no installer output)")
try:
    import radiomics
    RADIOMICS_AVAILABLE = True
    print("PyRadiomics available:", radiomics.__version__)
except Exception as exc:
    RADIOMICS_AVAILABLE = False
    print("PyRadiomics unavailable; segmentation API will still run without radiomics.")
    print(type(exc).__name__, str(exc))


(no installer output)
PyRadiomics available: 3.1.1.dev111+g8ed579383


In [6]:
# 4. Configure nnU-Net and verify the registered seg-model v1 checkpoint against the repository manifest
import hashlib

for key in ("nnUNet_raw", "nnUNet_preprocessed"):
    p = Path("/content/nnunet_work") / key
    p.mkdir(parents=True, exist_ok=True)
    os.environ[key] = str(p)
os.environ["nnUNet_results"] = str(BASE / "runs/M2-T04_nnunet/nnUNet_results")

manifest_path = REPO / "models/registry/seg-model/v1/manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError(f"Registered model manifest not found: {manifest_path}")
manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
checkpoint_rel = manifest.get("nnunet", {}).get("checkpoint", "checkpoint_best.pth")
checkpoint = MODEL_DIR / "fold_0" / checkpoint_rel
if not checkpoint.is_file():
    raise FileNotFoundError(f"Registered checkpoint missing: {checkpoint}. Complete M2-T04 setup first.")

h = hashlib.sha256()
with checkpoint.open("rb") as f:
    for block in iter(lambda: f.read(1 << 20), b""):
        h.update(block)
checkpoint_sha = h.hexdigest()
expected_sha = manifest.get("checkpoint_file", {}).get("sha256")
if expected_sha and checkpoint_sha != expected_sha:
    raise RuntimeError("Checkpoint SHA-256 mismatch: refusing to export an unverified model.")
print("Registered seg-model v1 checkpoint verified:", checkpoint.name)
print("Checkpoint SHA-256:", checkpoint_sha)


Registered seg-model v1 checkpoint verified: checkpoint_best.pth
Checkpoint SHA-256: 7dcb450bf5455a0c62c9e1b368ee6341c687f555528d5ed3cd3399d88ea0f5e0


In [14]:
# 5. Verify/reuse the TorchScript export or run export with full diagnostics
from bmts.deploy.export import sha256

export_manifest_path = EXPORT_DIR / "deploy_manifest.json"
export_ok = False
if export_manifest_path.is_file():
    try:
        em = json.loads(export_manifest_path.read_text(encoding="utf-8"))
        model_file = EXPORT_DIR / em["model_file"]
        export_ok = (
            em.get("source", {}).get("checkpoint_sha256") == checkpoint_sha
            and model_file.is_file()
            and sha256(model_file) == em.get("model_sha256")
        )
    except Exception as exc:
        print("Existing export manifest is invalid:", repr(exc))
        export_ok = False

if not export_ok:
    EXPORT_DIR.parent.mkdir(parents=True, exist_ok=True)
    cmd = [
        sys.executable, str(REPO / "scripts/export_model.py"),
        "--model-dir", str(MODEL_DIR), "--out", str(EXPORT_DIR)
    ]
    print("Exporting registered model for API inference...")
    export_result = subprocess.run(
        cmd, cwd=str(REPO), text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    print(export_result.stdout[-12000:] if export_result.stdout else "(export script produced no output)")
    if export_result.returncode != 0:
        raise RuntimeError(
            "TorchScript export failed (exit code {}). The complete exporter log is printed above. "
            "Do not continue to the API cells until this is resolved.".format(export_result.returncode)
        )
else:
    print("Verified TorchScript export already exists; reusing it.")

if not export_manifest_path.is_file():
    raise FileNotFoundError(
        f"Export script finished without creating the expected manifest: {export_manifest_path}"
    )
em = json.loads(export_manifest_path.read_text(encoding="utf-8"))
print("Export ready:", EXPORT_DIR / em["model_file"])


Verified TorchScript export already exists; reusing it.
Export ready: /content/drive/MyDrive/BrainMRI_Data/models/seg-model-v1-torchscript/seg_model.torchscript.pt


## 6. Write the FastAPI application

The backend uses `FastSegmenter` from the team's `bmts.deploy.segmenter` module, matching the M4 export notebook. It accepts only `.nii` / `.nii.gz` uploads, validates 3D images and matching geometry, and stores each request in its own folder to avoid overwriting concurrent uploads. Radiomics is optional: if extraction fails, the JSON reports that status without hiding a successful segmentation.


In [8]:
APP_SOURCE = r'''from __future__ import annotations

import json
import os
import re
import shutil
import uuid
from pathlib import Path
from typing import Any

import nibabel as nib
import numpy as np
import pandas as pd
from fastapi import FastAPI, File, Form, HTTPException, UploadFile
from fastapi.responses import JSONResponse

BASE = Path(os.getenv("BRAINMRI_BASE", "/content/drive/MyDrive/BrainMRI_Data"))
EXPORT_DIR = Path(os.getenv("BRAINMRI_EXPORT_DIR", str(BASE / "models/seg-model-v1-torchscript")))
OUTPUT_ROOT = Path(os.getenv("BRAINMRI_API_OUTPUT", str(BASE / "runs/API_Backend")))
DEVICE = os.getenv("BRAINMRI_DEVICE", "cuda" if __import__("torch").cuda.is_available() else "cpu")
RUN_RADIOMICS_DEFAULT = os.getenv("API_RUN_RADIOMICS", "true").lower() in {"1", "true", "yes"}
# TTA can multiply inference time substantially. It is disabled by default; set API_TTA=true to enable it.
TTA = os.getenv("API_TTA", "false").lower() in {"1", "true", "yes"}
MAX_UPLOAD_MB = int(os.getenv("API_MAX_UPLOAD_MB", "512"))
MODALITIES = ("t1", "t1ce", "t2", "flair")
REGIONS = {"TC": (1, 3), "WT": (1, 2, 3), "ET": (3,)}

app = FastAPI(
    title="Brain MRI Segmentation API",
    version="1.0.1",
    description="Project-compatible API for registered seg-model v1. Research prototype only."
)

_segmenter = None

def safe_id(value: str | None) -> str:
    value = (value or "case").strip()
    value = re.sub(r"[^A-Za-z0-9_.-]+", "_", value).strip("._-")
    return (value or "case")[:80]

def is_nii_name(filename: str | None) -> bool:
    name = (filename or "").lower()
    return name.endswith(".nii") or name.endswith(".nii.gz")

def get_segmenter():
    global _segmenter
    if _segmenter is None:
        if not EXPORT_DIR.is_dir() or not (EXPORT_DIR / "deploy_manifest.json").is_file():
            raise RuntimeError(f"Registered TorchScript export is missing: {EXPORT_DIR}")
        from bmts.deploy.segmenter import FastSegmenter
        _segmenter = FastSegmenter.from_dir(EXPORT_DIR, device=DEVICE, tta=TTA)
    return _segmenter

async def save_upload(upload: UploadFile, dest: Path) -> None:
    if not is_nii_name(upload.filename):
        raise HTTPException(status_code=415, detail=f"Unsupported file type for {upload.filename!r}; upload .nii or .nii.gz.")
    total = 0
    dest.parent.mkdir(parents=True, exist_ok=True)
    try:
        with dest.open("wb") as out:
            while True:
                chunk = await upload.read(1024 * 1024)
                if not chunk:
                    break
                total += len(chunk)
                if total > MAX_UPLOAD_MB * 1024 * 1024:
                    raise HTTPException(status_code=413, detail=f"File exceeds {MAX_UPLOAD_MB} MB limit.")
                out.write(chunk)
    finally:
        await upload.close()
    if total == 0:
        dest.unlink(missing_ok=True)
        raise HTTPException(status_code=400, detail=f"Empty upload: {upload.filename!r}")

def validate_modalities(paths: dict[str, Path]) -> dict[str, Any]:
    images = {}
    try:
        for name in MODALITIES:
            p = paths[name]
            try:
                img = nib.load(str(p))
                if len(img.shape) != 3:
                    raise ValueError(f"{name} must be a 3D NIfTI image; got shape {img.shape}.")
                if not np.all(np.isfinite(img.affine)):
                    raise ValueError(f"{name} has a non-finite affine matrix.")
                images[name] = img
            except Exception as exc:
                raise ValueError(f"Invalid or unreadable NIfTI for {name}: {exc}") from exc
        ref = images["t1"]
        for name, img in images.items():
            if img.shape[:3] != ref.shape[:3] or not np.allclose(img.affine, ref.affine, atol=1e-4):
                raise ValueError(f"Geometry mismatch: {name} shape/affine differs from t1.")
        voxel_volume_mm3 = float(abs(np.linalg.det(ref.affine[:3, :3])))
        return {
            "shape": [int(x) for x in ref.shape],
            "voxel_volume_mm3": voxel_volume_mm3,
            "voxel_spacing_mm": [float(x) for x in ref.header.get_zooms()[:3]],
            "geometry_validated": True,
        }
    finally:
        images.clear()

def normalize_segmentation_axes(seg: np.ndarray, reference_shape: tuple[int, ...]) -> np.ndarray:
    """Return the prediction in the reference NIfTI array order (i, j, k).

    Some inference wrappers return arrays in z,y,x order. Only reverse all axes
    when the shape exactly matches the reversed reference shape; never guess for
    other shapes.
    """
    arr = np.asarray(seg)
    target = tuple(int(x) for x in reference_shape[:3])
    if arr.shape == target:
        return arr
    if arr.ndim == 3 and arr.shape == target[::-1]:
        return np.transpose(arr, (2, 1, 0))
    raise ValueError(f"Prediction shape {arr.shape} does not match input NIfTI shape {target} or its reversed order.")

def calculate_volumes(mask_path: Path, voxel_volume_mm3: float) -> dict[str, Any]:
    mask_img = nib.load(str(mask_path))
    mask = np.asarray(mask_img.dataobj)
    if mask.ndim != 3:
        raise ValueError("Predicted mask is not 3D.")
    result = {}
    for region, labels in REGIONS.items():
        voxels = int(np.isin(mask, labels).sum())
        result[region] = {
            "voxels": voxels,
            "volume_cm3": round(voxels * voxel_volume_mm3 / 1000.0, 6),
        }
    result["label_counts"] = {str(int(label)): int((mask == label).sum()) for label in np.unique(mask)}
    return result

def extract_radiomics(paths: dict[str, Path], mask_path: Path, out_csv: Path) -> dict[str, Any]:
    try:
        from radiomics import featureextractor
        import SimpleITK as sitk

        mask = sitk.ReadImage(str(mask_path))
        settings = {
            "binWidth": 25,
            "resampledPixelSpacing": None,
            "interpolator": "sitkBSpline",
            "normalize": False,
            "label": 1,
        }
        extractor = featureextractor.RadiomicsFeatureExtractor(**settings)
        rows = []
        for modality in ("t1ce", "t2", "flair"):
            image = sitk.ReadImage(str(paths[modality]))
            # Explicitly check dimensions and physical geometry; do not silently resize/transpose.
            if image.GetSize() != mask.GetSize():
                raise ValueError(
                    f"Radiomics geometry mismatch for {modality}: image size {image.GetSize()} "
                    f"vs mask size {mask.GetSize()}. Both files must be in the same NIfTI geometry "
                    "and must be read from the current request folder."
                )
            if (image.GetOrigin() != mask.GetOrigin() or image.GetSpacing() != mask.GetSpacing()
                    or image.GetDirection() != mask.GetDirection()):
                raise ValueError(f"Radiomics physical geometry mismatch for {modality} image and mask.")
            for label in (1, 2, 3):
                binary = sitk.Cast(mask == label, sitk.sitkUInt8)
                if int(sitk.GetArrayViewFromImage(binary).sum()) == 0:
                    continue
                features = extractor.execute(image, binary, label=1)
                row = {"modality": modality, "label": label}
                row.update({str(k): (float(v) if isinstance(v, (int, float, np.number)) else str(v))
                            for k, v in features.items() if not str(k).startswith("diagnostics_")})
                rows.append(row)
        if not rows:
            return {"status": "empty_mask", "csv_path": None, "rows": 0}
        out_csv.parent.mkdir(parents=True, exist_ok=True)
        pd.DataFrame(rows).to_csv(out_csv, index=False)
        return {"status": "success", "csv_path": str(out_csv), "rows": len(rows)}
    except Exception as exc:
        return {"status": "failed", "error": f"{type(exc).__name__}: {exc}", "csv_path": None}

def run_one_case(case_id: str, paths: dict[str, Path], case_out: Path, run_radiomics: bool) -> dict[str, Any]:
    validation = validate_modalities(paths)
    case_out.mkdir(parents=True, exist_ok=True)
    mask_path = case_out / f"{case_id}_seg.nii.gz"
    try:
        segmenter = get_segmenter()
        # FastSegmenter expects file paths. Pass plain strings rather than Path objects
        # to avoid path-like tuple/type incompatibilities across repository versions.
        input_files = [str(paths[m]) for m in MODALITIES]
        segment_output = segmenter.segment(input_files)
        if not isinstance(segment_output, tuple) or len(segment_output) < 3:
            raise TypeError(
                "Unexpected FastSegmenter.segment() result. Expected (segmentation, output, timings); "
                f"received {type(segment_output).__name__}."
            )
        seg, _segment_output_path, timings = segment_output
        if isinstance(seg, (str, Path)):
            # Some versions return a saved mask path as the first result.
            seg = np.asarray(nib.load(str(seg)).dataobj)
        elif isinstance(seg, tuple):
            # Do not pass tuple objects to Path/nibabel; only accept a tuple that wraps one array.
            if len(seg) == 1:
                seg = seg[0]
            else:
                raise TypeError(
                    "FastSegmenter returned a tuple for the segmentation array; "
                    f"tuple length={len(seg)}. Check the installed segmenter API."
                )
        ref = nib.load(str(paths["t1"]))
        seg = normalize_segmentation_axes(np.asarray(seg), ref.shape)
        seg_img = nib.Nifti1Image(seg.astype(np.uint8), ref.affine, ref.header.copy())
        nib.save(seg_img, str(mask_path))
    except HTTPException:
        raise
    except Exception as exc:
        raise RuntimeError(f"Segmentation failed: {type(exc).__name__}: {exc}") from exc

    volumes = calculate_volumes(mask_path, validation["voxel_volume_mm3"])
    radiomics = (extract_radiomics(paths, mask_path, case_out / f"{case_id}_radiomics.csv")
                 if run_radiomics else {"status": "disabled", "csv_path": None, "rows": 0})
    return {
        "case_id": case_id,
        "input_validation": validation,
        "segmentation": {"status": "success", "mask_path": str(mask_path), "volumes": volumes},
        "inference_timings": {str(k): float(v) for k, v in timings.items()},
        "radiomics": radiomics,
    }

@app.get("/health")
def health():
    return {
        "status": "ok",
        "service": "Brain MRI Segmentation API",
        "export_dir_exists": EXPORT_DIR.is_dir(),
        "export_manifest_exists": (EXPORT_DIR / "deploy_manifest.json").is_file(),
        "device": DEVICE,
        "tta_enabled": TTA,
        "supported_file_extensions": [".nii", ".nii.gz"],
        "modalities_required": list(MODALITIES),
    }

@app.post("/predict")
async def predict(
    t1: UploadFile = File(...),
    t1ce: UploadFile = File(...),
    t2: UploadFile = File(...),
    flair: UploadFile = File(...),
    case_id: str = Form("case"),
    run_radiomics: bool = Form(RUN_RADIOMICS_DEFAULT),
):
    cid = safe_id(case_id)
    request_dir = OUTPUT_ROOT / "requests" / f"{cid}_{uuid.uuid4().hex[:8]}"
    inputs_dir = request_dir / "inputs"
    uploads = {"t1": t1, "t1ce": t1ce, "t2": t2, "flair": flair}
    paths = {}
    try:
        for modality, upload in uploads.items():
            suffix = ".nii.gz" if (upload.filename or "").lower().endswith(".nii.gz") else ".nii"
            paths[modality] = inputs_dir / f"{cid}_{modality}{suffix}"
            await save_upload(upload, paths[modality])
        result = run_one_case(cid, paths, request_dir / "results", run_radiomics)
        return JSONResponse({"status": "success", "pipeline": "seg-model-v1-torchscript",
                             "result": result, "longitudinal_changes": None})
    except HTTPException:
        shutil.rmtree(request_dir, ignore_errors=True)
        raise
    except ValueError as exc:
        shutil.rmtree(request_dir, ignore_errors=True)
        raise HTTPException(status_code=422, detail=str(exc)) from exc
    except RuntimeError as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"{type(exc).__name__}: {exc}") from exc

@app.post("/predict/longitudinal")
async def predict_longitudinal(
    patient_id: str = Form(...),
    visits_json: str = Form(..., description='JSON array: [{"visit_id":"preop","t1":"filename.nii.gz","t1ce":"...","t2":"...","flair":"..."}]'),
    files: list[UploadFile] = File(..., description="Upload 4 files per visit, ordered by visits_json: t1, t1ce, t2, flair."),
    run_radiomics: bool = Form(RUN_RADIOMICS_DEFAULT),
):
    """Predict each visit for one patient. Files must be ordered visit-by-visit, 4 modalities per visit."""
    try:
        visits = json.loads(visits_json)
    except Exception as exc:
        raise HTTPException(status_code=422, detail="visits_json must be valid JSON.") from exc
    if not isinstance(visits, list) or len(visits) < 2:
        raise HTTPException(status_code=422, detail="At least two visits are required.")
    if len(files) != 4 * len(visits):
        raise HTTPException(status_code=422, detail=f"Expected {4 * len(visits)} files (4 modalities per visit), received {len(files)}.")
    patient = safe_id(patient_id)
    request_dir = OUTPUT_ROOT / "longitudinal" / f"{patient}_{uuid.uuid4().hex[:8]}"
    outputs = []
    try:
        for i, visit in enumerate(visits):
            if not isinstance(visit, dict):
                raise HTTPException(status_code=422, detail=f"Visit {i} must be a JSON object.")
            visit_id = safe_id(str(visit.get("visit_id", f"visit_{i+1}")))
            visit_dir = request_dir / visit_id
            input_dir = visit_dir / "inputs"
            input_dir.mkdir(parents=True, exist_ok=True)
            paths = {}
            for j, modality in enumerate(MODALITIES):
                upload = files[i * 4 + j]
                if not is_nii_name(upload.filename):
                    raise HTTPException(status_code=415, detail=f"Unsupported file type for {upload.filename!r}; upload .nii or .nii.gz.")
                suffix = ".nii.gz" if (upload.filename or "").lower().endswith(".nii.gz") else ".nii"
                path = input_dir / f"{visit_id}_{modality}{suffix}"
                await save_upload(upload, path)
                paths[modality] = path
            result = run_one_case(f"{patient}_{visit_id}", paths, visit_dir / "results", run_radiomics)
            outputs.append({"visit_id": visit_id, **result})
        changes = []
        for before, after in zip(outputs, outputs[1:]):
            for region in ("TC", "WT", "ET"):
                v0 = before["segmentation"]["volumes"][region]["volume_cm3"]
                v1 = after["segmentation"]["volumes"][region]["volume_cm3"]
                pct = None if v0 == 0 else round((v1 - v0) / v0 * 100.0, 3)
                changes.append({
                    "from_visit": before["visit_id"], "to_visit": after["visit_id"],
                    "region": region, "volume_before_cm3": v0, "volume_after_cm3": v1,
                    "change_cm3": round(v1 - v0, 6), "change_percent": pct,
                    "note": "change_percent is null when baseline volume is zero"
                })
        return JSONResponse({"status": "success", "patient_id": patient,
                             "pipeline": "seg-model-v1-torchscript",
                             "visits": outputs, "longitudinal_changes": changes})
    except HTTPException:
        raise
    except ValueError as exc:
        raise HTTPException(status_code=422, detail=str(exc)) from exc
    except Exception as exc:
        raise HTTPException(status_code=500, detail=f"{type(exc).__name__}: {exc}") from exc

@app.get("/")
def root():
    return {"message": "Brain MRI API is running", "docs": "/docs", "health": "/health"}
'''
app_path = API_DIR / 'main.py'
app_path.write_text(APP_SOURCE, encoding='utf-8')
print('Wrote:', app_path)
print('Bytes:', app_path.stat().st_size)


Wrote: /content/drive/MyDrive/BrainMRI_Data/api_backend/main.py
Bytes: 15926


## 7. Add dependencies, environment template, and README

In [9]:
# Create deployment metadata and dependencies.
# PyRadiomics is intentionally optional: Colab/Python 3.13 may fail to build the PyPI source distribution.
requirements = '''fastapi>=0.115,<1.0
uvicorn[standard]>=0.30,<1.0
python-multipart
nnunetv2==2.8.1
nibabel>=5.2
numpy>=1.26
pandas>=2.0
SimpleITK>=2.2.1
'''
(API_DIR / "requirements.txt").write_text(requirements, encoding="utf-8")

env_example = '''BRAINMRI_BASE=/content/drive/MyDrive/BrainMRI_Data
BRAINMRI_EXPORT_DIR=/content/drive/MyDrive/BrainMRI_Data/models/seg-model-v1-torchscript
BRAINMRI_API_OUTPUT=/content/drive/MyDrive/BrainMRI_Data/runs/API_Backend
BRAINMRI_DEVICE=cpu
API_RUN_RADIOMICS=true
API_MAX_UPLOAD_MB=512
'''
(API_DIR / ".env.example").write_text(env_example, encoding="utf-8")

readme = '''# Brain MRI FastAPI Backend (project-compatible)

Uses the registered seg-model v1 TorchScript export and the project label scheme.

## Run in Colab
Run the notebook from top to bottom. It mounts Drive, verifies the registered checkpoint, reuses/creates the TorchScript export, writes the backend, starts Uvicorn, and checks the health endpoint with a startup timeout and captured logs.

## Endpoints
- `GET /health`
- `POST /predict`: file fields `t1`, `t1ce`, `t2`, `flair`, plus `case_id` and `run_radiomics`.
- `POST /predict/longitudinal`: `patient_id`, `visits_json`, and four files per visit ordered by visit and modality.

## Input requirements
Only `.nii` and `.nii.gz` files are accepted. Inputs must be readable 3D NIfTI images with matching dimensions and affine geometry. Upload cleaned/standardized modality files.

## Outputs
Masks, TC/WT/ET volumes, inference timings, optional radiomics CSV, and sequential longitudinal volume changes are saved below `runs/API_Backend/`.

## Limitations
- The model is not retrained.
- Longitudinal percentage change is undefined when the baseline volume is zero.
- Longitudinal scans should be registered/aligned.
- Radiomics is optional and its failure does not suppress a successful segmentation.
- Research prototype only; not for clinical diagnosis.
'''
(API_DIR / "README.md").write_text(readme, encoding="utf-8")
print("Created requirements.txt, .env.example, README.md")


Created requirements.txt, .env.example, README.md


## 8. Validate the application code before starting the server

In [11]:
import ast
ast.parse(app_path.read_text(encoding="utf-8"))
print("Python syntax validation: PASS")
print("Backend files:", sorted(p.name for p in API_DIR.iterdir()))


Python syntax validation: PASS
Backend files: ['.env.example', 'README.md', '__pycache__', 'main.py', 'requirements.txt']


## 9. Start the API server in the background

After this cell finishes, open the interactive docs at the local Colab URL shown by the proxy cell below. The server uses the project Drive export and saves results under `runs/API_Backend/`.


In [10]:
# Start Uvicorn robustly, keep logs, and wait for /health to respond.
import os, sys, time, subprocess, requests
from pathlib import Path

BASE_URL = "http://127.0.0.1:8000"
SERVER_LOG = Path("/content/brainmri_api.log")

# Stop only a server process started by this notebook.
if "api_process" in globals() and api_process is not None and api_process.poll() is None:
    api_process.terminate()
    try:
        api_process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        api_process.kill()
        api_process.wait(timeout=5)

if "server_log_handle" in globals():
    try:
        server_log_handle.close()
    except Exception:
        pass

server_log_handle = open(SERVER_LOG, "w", buffering=1)
server_env = os.environ.copy()
server_env.update({
    "BRAINMRI_BASE": str(BASE),
    "BRAINMRI_EXPORT_DIR": str(EXPORT_DIR),
    "BRAINMRI_API_OUTPUT": str(OUT_DIR),
    # The exported TorchScript was generated with CPU Torch; use CPU by default for reliable startup.
    "BRAINMRI_DEVICE": "cpu",
    "PYTHONUNBUFFERED": "1",
    "PYTHONPATH": str(REPO / "src") + os.pathsep + os.environ.get("PYTHONPATH", ""),
})

api_process = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app",
     "--host", "0.0.0.0", "--port", "8000", "--log-level", "info"],
    cwd=str(API_DIR),
    stdout=server_log_handle,
    stderr=subprocess.STDOUT,
    env=server_env,
)

print("Started Uvicorn PID:", api_process.pid)
health_ok = False
last_error = None

for attempt in range(60):
    if api_process.poll() is not None:
        print("❌ Uvicorn exited with code:", api_process.returncode)
        break
    try:
        r = requests.get(f"{BASE_URL}/health", timeout=2)
        if r.status_code == 200:
            print("✅ GET /health:", r.status_code)
            print(r.json())
            health_ok = True
            break
        last_error = f"HTTP {r.status_code}: {r.text[:1000]}"
    except requests.RequestException as exc:
        last_error = f"{type(exc).__name__}: {exc}"
    time.sleep(1)

if not health_ok:
    server_log_handle.flush()
    print("\n❌ API health check did not succeed.")
    print("Last connection result:", last_error)
    print("\n--- Uvicorn log (last 15000 chars) ---")
    print(SERVER_LOG.read_text(errors="replace")[-15000:] if SERVER_LOG.exists() else "(log file missing)")
    raise RuntimeError("FastAPI did not become healthy. Diagnose the Uvicorn log above before continuing.")

print("API server is healthy and listening on port 8000.")


Started Uvicorn PID: 3132
✅ GET /health: 200
{'status': 'ok', 'service': 'Brain MRI Segmentation API', 'export_dir_exists': True, 'export_manifest_exists': True, 'device': 'cpu', 'tta_enabled': False, 'supported_file_extensions': ['.nii', '.nii.gz'], 'modalities_required': ['t1', 't1ce', 't2', 'flair']}
API server is healthy and listening on port 8000.


## 10. Smoke-test health and wrong file type

This test checks the health endpoint and confirms that a `.txt` upload is rejected with HTTP 415. It does not run a full segmentation, because that requires four real cleaned MRI modality files.


In [12]:
# Smoke-test health and unsupported file-type rejection; show server logs if a request fails.
import requests
from pathlib import Path

BASE_URL = "http://127.0.0.1:8000"
try:
    health_response = requests.get(f"{BASE_URL}/health", timeout=10)
    print("GET /health:", health_response.status_code, health_response.text[:2000])
    health_response.raise_for_status()

    bad = {
        "t1": ("t1.txt", b"not a nifti", "text/plain"),
        "t1ce": ("t1ce.nii.gz", b"placeholder", "application/octet-stream"),
        "t2": ("t2.nii.gz", b"placeholder", "application/octet-stream"),
        "flair": ("flair.nii.gz", b"placeholder", "application/octet-stream"),
    }
    bad_response = requests.post(
        f"{BASE_URL}/predict", files=bad,
        data={"case_id": "invalid_type", "run_radiomics": "false"}, timeout=30
    )
    print("Wrong file type status:", bad_response.status_code)
    print("Response:", bad_response.text[:1000])
    assert bad_response.status_code == 415, "API should reject unsupported file types with HTTP 415."
    print("API smoke tests: PASS")
except Exception:
    log_path = Path("/content/brainmri_api.log")
    print("\n--- Uvicorn log (last 15000 chars) ---")
    print(log_path.read_text(errors="replace")[-15000:] if log_path.exists() else "(log file missing)")
    raise


GET /health: 200 {"status":"ok","service":"Brain MRI Segmentation API","export_dir_exists":true,"export_manifest_exists":true,"device":"cpu","tta_enabled":false,"supported_file_extensions":[".nii",".nii.gz"],"modalities_required":["t1","t1ce","t2","flair"]}
Wrong file type status: 415
Response: {"detail":"Unsupported file type for 't1.txt'; upload .nii or .nii.gz."}
API smoke tests: PASS


## 11. Access the interactive docs

In Colab, use the proxy URL for port 8000. The Swagger page lets you upload the four modality files to `/predict`. For a public/shareable deployment, deploy the same `api_backend` folder to a server with the model export and dependencies; a local Colab runtime is temporary.



In [13]:
# Display clickable links directly in the notebook output.
from google.colab.output import eval_js
from IPython.display import display, HTML

BASE_URL = eval_js("google.colab.kernel.proxyPort(8000)")
DOCS_URL = BASE_URL.rstrip("/") + "/docs"
HEALTH_URL = BASE_URL.rstrip("/") + "/health"

print("FastAPI backend is running.")
display(HTML(f'''
<div style="padding:14px;border:1px solid #cbd5e1;border-radius:10px;line-height:2">
  <h3 style="margin:0 0 8px 0">Brain MRI FastAPI Backend</h3>
  <div>🌐 <b>Open API website:</b> <a href="{BASE_URL}" target="_blank">{BASE_URL}</a></div>
  <div>🧪 <b>API documentation (Swagger UI):</b> <a href="{DOCS_URL}" target="_blank">{DOCS_URL}</a></div>
  <div>💚 <b>Health check:</b> <a href="{HEALTH_URL}" target="_blank">{HEALTH_URL}</a></div>
</div>
'''))
print("Note: these links work while this Google Colab runtime and the API server remain active.")


FastAPI backend is running.


Note: these links work while this Google Colab runtime and the API server remain active.


In [15]:
import requests

response = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=15
)

print("Status Code:", response.status_code)
print("Response:")
print(response.json())

Status Code: 200
Response:
{'status': 'ok', 'service': 'Brain MRI Segmentation API', 'export_dir_exists': True, 'export_manifest_exists': True, 'device': 'cpu', 'tta_enabled': False, 'supported_file_extensions': ['.nii', '.nii.gz'], 'modalities_required': ['t1', 't1ce', 't2', 'flair']}


In [16]:
from google.colab.output import eval_js

BASE_URL = eval_js(
    "google.colab.kernel.proxyPort(8000)"
)

print("API URL:", BASE_URL)
print("Health URL:", BASE_URL.rstrip("/") + "/health")
print("Docs URL:", BASE_URL.rstrip("/") + "/docs")

API URL: https://8000-m-s-use1c9-61rv1fomwuxw-c.us-east1.prod.r.colab.dev
Health URL: https://8000-m-s-use1c9-61rv1fomwuxw-c.us-east1.prod.r.colab.dev/health
Docs URL: https://8000-m-s-use1c9-61rv1fomwuxw-c.us-east1.prod.r.colab.dev/docs


In [17]:
from pathlib import Path

log = Path("/content/brainmri_api.log")

if log.exists():
    print(log.read_text(errors="replace")[-10000:])
else:
    print("Server log not found.")

INFO:     Started server process [3132]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:32816 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:56984 - "GET /health HTTP/1.1" 200 OK
INFO:     127.0.0.1:56996 - "POST /predict HTTP/1.1" 415 Unsupported Media Type
INFO:     127.0.0.1:60390 - "GET /health HTTP/1.1" 200 OK



In [18]:
from google.colab.output import eval_js
from IPython.display import IFrame, display

proxy_base = eval_js(
    "google.colab.kernel.proxyPort(8000)"
).rstrip("/")

health_url = proxy_base + "/health"
docs_url = proxy_base + "/docs"

print("Health URL:", health_url)
print("Docs URL:", docs_url)

print("Testing Health inside Colab:")

display(
    IFrame(
        src=health_url,
        width="100%",
        height=150
    )
)

print("Opening Swagger UI:")

display(
    IFrame(
        src=docs_url,
        width="100%",
        height=700
    )
)

Health URL: https://8000-m-s-use1c9-61rv1fomwuxw-c.us-east1.prod.r.colab.dev/health
Docs URL: https://8000-m-s-use1c9-61rv1fomwuxw-c.us-east1.prod.r.colab.dev/docs
Testing Health inside Colab:


Opening Swagger UI:


In [19]:
from google.colab import files
from pathlib import Path

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Please upload exactly one file.")

file_name, content = next(iter(uploaded.items()))

if not file_name.endswith("web_interface.py"):
    raise ValueError("Please select web_interface.py")

target = Path("/content/repo/apps/web_interface.py")
target.parent.mkdir(parents=True, exist_ok=True)

target.write_bytes(content)

print("File saved successfully!")
print("Location:", target)
print("File size:", target.stat().st_size, "bytes")

Saving web_interface.py to web_interface.py
File saved successfully!
Location: /content/repo/apps/web_interface.py
File size: 4074 bytes


In [20]:
from pathlib import Path
import ast
import importlib.util
import requests

# 1. Check frontend file
app_path = Path("/content/repo/apps/web_interface.py")

print("Frontend file exists:", app_path.exists())

if not app_path.exists():
    raise FileNotFoundError("web_interface.py not found")

# 2. Check Python syntax
source = app_path.read_text(encoding="utf-8")
ast.parse(source)

print("Python syntax: OK")
print("File size:", app_path.stat().st_size, "bytes")

# 3. Check Streamlit
streamlit_installed = (
    importlib.util.find_spec("streamlit") is not None
)

print("Streamlit installed:", streamlit_installed)

# 4. Check backend
response = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=15
)

print("Backend status:", response.status_code)
print("Backend response:", response.json())

Frontend file exists: True
Python syntax: OK
File size: 4074 bytes
Streamlit installed: False
Backend status: 200
Backend response: {'status': 'ok', 'service': 'Brain MRI Segmentation API', 'export_dir_exists': True, 'export_manifest_exists': True, 'device': 'cpu', 'tta_enabled': False, 'supported_file_extensions': ['.nii', '.nii.gz'], 'modalities_required': ['t1', 't1ce', 't2', 'flair']}


In [21]:
%pip install -q streamlit==1.66.1

In [22]:
import streamlit as st

print("Streamlit installed successfully!")
print("Version:", st.__version__)

Streamlit installed successfully!
Version: 1.66.1


In [23]:
import subprocess
import sys
import time
import requests
from pathlib import Path

APP_PATH = Path("/content/repo/apps/web_interface.py")
LOG_PATH = Path("/content/streamlit.log")

# Make sure the frontend file exists
assert APP_PATH.exists(), "web_interface.py not found!"

# Check that the backend is still running
backend = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=15
)

assert backend.status_code == 200

print("Backend: READY")

# Start Streamlit
log_file = open(LOG_PATH, "w")

streamlit_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "streamlit",
        "run",
        str(APP_PATH),
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--browser.gatherUsageStats=false",
    ],
    cwd="/content/repo",
    stdout=log_file,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

# Wait for Streamlit
streamlit_ready = False

for attempt in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=2
        )

        if response.status_code == 200:
            streamlit_ready = True
            break

    except requests.RequestException:
        time.sleep(1)

if streamlit_ready:
    print("Streamlit: READY")
    print("Frontend is running on port 8501!")
else:
    print("Streamlit failed to start.")
    print(LOG_PATH.read_text(errors="replace")[-5000:])

Backend: READY
Streamlit: READY
Frontend is running on port 8501!


In [27]:
from google.colab import output

output.serve_kernel_port_as_iframe(
    8501,
    path="/",
    height=900
)

<IPython.core.display.Javascript object>

In [28]:
import requests
from pathlib import Path

# 1. Check Streamlit main page
try:
    r = requests.get(
        "http://127.0.0.1:8501/",
        timeout=10
    )
    print("Homepage status:", r.status_code)
    print("Content-Type:", r.headers.get("content-type"))
    print("HTML received:", "streamlit" in r.text.lower())
except Exception as e:
    print("Homepage error:", repr(e))


# 2. Check Streamlit internal health
try:
    r = requests.get(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=10
    )
    print("Streamlit health:", r.status_code)
except Exception as e:
    print("Health error:", repr(e))


# 3. Read Streamlit logs
log_path = Path("/content/streamlit.log")

print("\n========== STREAMLIT LOGS ==========")

if log_path.exists():
    print(
        log_path.read_text(errors="replace")[-8000:]
    )
else:
    print("Log file not found.")


Homepage status: 200
Content-Type: text/html; charset=utf-8
HTML received: True
Streamlit health: 200

========== STREAMLIT LOGS ==========
od.r.colab.dev, host=m-s-use1c9-61rv1fomwuxw.us-east1-c.c.codatalab-user-runtimes.internal:8007
2026-11-02 09:24:27.670 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-use1c9-61rv1fomwuxw-c.us-east1.prod.r.colab.dev, host=m-s-use1c9-61rv1fomwuxw.us-east1-c.c.codatalab-user-runtimes.internal:8007
2026-11-02 09:24:32.437 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-use1c9-61rv1fomwuxw-c.us-east1.prod.r.colab.dev, host=m-s-use1c9-61rv1fomwuxw.us-east1-c.c.codatalab-user-runtimes.internal:8007
2026-11-02 09:24:37.109 Rejecting WebSocket connection with disallowed Origin or Host header: origin=https://8501-m-s-use1c9-61rv1fomwuxw-c.us-east1.prod.r.colab.dev, host=m-s-use1c9-61rv1fomwuxw.us-east1-c.c.codatalab-user-runtimes.internal:8007
2026-11-02 09:24:42.033

In [29]:

import subprocess
import sys
import time
import requests
from pathlib import Path

# 1. Stop the old Streamlit process
old_process = globals().get("streamlit_process")

if old_process is not None and old_process.poll() is None:
    old_process.terminate()

    try:
        old_process.wait(timeout=10)
    except subprocess.TimeoutExpired:
        old_process.kill()
        old_process.wait()

print("Old Streamlit process stopped.")

# 2. Prepare a fresh log file
LOG_PATH = Path("/content/streamlit.log")
APP_PATH = Path("/content/repo/apps/web_interface.py")

assert APP_PATH.exists(), "Frontend file not found!"

# 3. Start Streamlit with Colab proxy compatibility
with open(LOG_PATH, "w") as log_file:
    streamlit_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            str(APP_PATH),
            "--server.port=8501",
            "--server.address=0.0.0.0",
            "--server.headless=true",
            "--server.enableCORS=false",
            "--server.enableXsrfProtection=false",
            "--server.allowedHosts=*",
            "--browser.gatherUsageStats=false",
        ],
        cwd="/content/repo",
        stdout=log_file,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

# 4. Wait for Streamlit
ready = False

for _ in range(40):
    try:
        response = requests.get(
            "http://127.0.0.1:8501/_stcore/health",
            timeout=3,
        )

        if response.status_code == 200:
            ready = True
            break

    except requests.RequestException:
        pass

    time.sleep(1)

print("Streamlit READY:", ready)

# 5. Check backend
backend = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=15,
)

print("Backend status:", backend.status_code)

if not ready:
    print(LOG_PATH.read_text(errors="replace")[-5000:])


Old Streamlit process stopped.
Streamlit READY: True
Backend status: 200


In [30]:
from google.colab import output

output.serve_kernel_port_as_iframe(
    8501,
    path="/",
    height=900
)

<IPython.core.display.Javascript object>

In [31]:

from pathlib import Path
import nibabel as nib
import numpy as np

BASE = Path("/content/drive/MyDrive/BrainMRI_Data")

search_roots = [
    Path("/content/processed"),
    BASE / "processed",
    BASE / "data" / "processed",
]

# Use the project Drive root as a fallback.
if BASE.is_dir():
    search_roots.append(BASE)

found_case = None

def get_modality(folder, name):
    for ext in (".nii.gz", ".nii"):
        candidate = folder / f"{name}{ext}"
        if candidate.is_file():
            return candidate
    return None

for root in search_roots:
    if not root.is_dir():
        continue

    print("Searching:", root)

    for t1_path in list(root.rglob("t1.nii.gz")) + list(root.rglob("t1.nii")):
        folder = t1_path.parent

        paths = {
            name: get_modality(folder, name)
            for name in ("t1", "t1ce", "t2", "flair")
        }

        if not all(paths.values()):
            continue

        print("\nCandidate case:", folder)

        images = {
            name: nib.load(str(path))
            for name, path in paths.items()
        }

        reference = images["t1"]
        shape_ok = all(
            len(img.shape) == 3 and img.shape == reference.shape
            for img in images.values()
        )

        affine_ok = all(
            np.allclose(
                img.affine,
                reference.affine,
                atol=1e-4
            )
            for img in images.values()
        )

        print("Shape:", reference.shape)
        print("Matching dimensions:", shape_ok)
        print("Matching affine:", affine_ok)

        for name, path in paths.items():
            print(f"{name}: {path.name}")

        if shape_ok and affine_ok:
            found_case = {
                "folder": folder,
                "paths": paths,
                "shape": reference.shape,
            }
            break

    if found_case:
        break

if found_case:
    print("\nSUCCESS: A compatible MRI case was found.")
else:
    print("\nNo compatible cleaned MRI case was found.")


Searching: /content/drive/MyDrive/BrainMRI_Data/processed

Candidate case: /content/drive/MyDrive/BrainMRI_Data/processed/brats2021/BraTS2021_00110/0
Shape: (240, 240, 155)
Matching dimensions: True
Matching affine: True
t1: t1.nii.gz
t1ce: t1ce.nii.gz
t2: t2.nii.gz
flair: flair.nii.gz

SUCCESS: A compatible MRI case was found.


In [32]:

import requests
import json

API = "http://127.0.0.1:8000"

# Get the API specification
response = requests.get(
    f"{API}/openapi.json",
    timeout=15
)

response.raise_for_status()
spec = response.json()

# Inspect POST /predict
endpoint = spec["paths"]["/predict"]["post"]

print("Endpoint: POST /predict")
print("\nSummary:")
print(endpoint.get("summary", "Not specified"))

# Inspect request body
request_body = endpoint.get("requestBody", {})
content = request_body.get("content", {})

print("\nAccepted Content Types:")
for content_type in content:
    print("-", content_type)

schema = content.get(
    "multipart/form-data", {}
).get("schema", {})

# Resolve schema references
if "$ref" in schema:
    ref_name = schema["$ref"].split("/")[-1]
    schema = spec["components"]["schemas"][ref_name]

properties = schema.get("properties", {})
required = schema.get("required", [])

print("\nExpected Request Fields:")

for field, details in properties.items():
    print(
        f"- {field}: "
        f"type={details.get('type')}, "
        f"format={details.get('format')}, "
        f"required={field in required}"
    )

print("\nResponse Codes:")

for code, details in endpoint.get("responses", {}).items():
    print("-", code, ":", details.get("description"))

print("\nAPI specification retrieved successfully!")


Endpoint: POST /predict

Summary:
Predict

Accepted Content Types:
- multipart/form-data

Expected Request Fields:
- t1: type=string, format=None, required=True
- t1ce: type=string, format=None, required=True
- t2: type=string, format=None, required=True
- flair: type=string, format=None, required=True
- case_id: type=string, format=None, required=False
- run_radiomics: type=boolean, format=None, required=False

Response Codes:
- 200 : Successful Response
- 422 : Validation Error

API specification retrieved successfully!


In [37]:

import requests
import json
import time
import nibabel as nib
import numpy as np

from pathlib import Path

# 1. API configuration
API_URL = "http://127.0.0.1:8000"

CASE_DIR = Path(
    "/content/drive/MyDrive/BrainMRI_Data/processed/"
    "brats2021/BraTS2021_00110/0"
)

# 2. Check input files
modalities = ["t1", "t1ce", "t2", "flair"]

paths = {
    name: CASE_DIR / f"{name}.nii.gz"
    for name in modalities
}

for name, path in paths.items():
    assert path.exists(), f"Missing file: {path}"
    print(f"{name}: {path.stat().st_size / (1024**2):.2f} MB")

# 3. Check backend
health = requests.get(
    f"{API_URL}/health",
    timeout=15
)

assert health.status_code == 200

print("\nBackend is ready.")

# 4. Prepare files for API
file_handles = {}

try:
    for name, path in paths.items():
        file_handles[name] = (
            path.name,
            open(path, "rb"),
            "application/octet-stream",
        )

    print("\nSending MRI files to the AI model...")
    print("Radiomics: disabled for this test")

    start_time = time.time()

    response = requests.post(
        f"{API_URL}/predict",
        files=file_handles,
        data={
            "case_id": "BraTS2021_00110",
            "run_radiomics": "false",
        },
        timeout=(30, 1800),
    )

    elapsed = time.time() - start_time

finally:
    for _, handle, _ in file_handles.values():
        handle.close()

# 5. Inspect response
print("\nHTTP Status:", response.status_code)

if response.status_code != 200:
    print("API Error:")
    print(response.text[:5000])

else:
    data = response.json()
    result = data["result"]

    print("\nSegmentation Status:",
          result["segmentation"]["status"])

    print("Case ID:", result["case_id"])

    print("\nInput Validation:")
    print(json.dumps(
        result["input_validation"],
        indent=2
    ))

    print("\nTumor Measurements:")
    print(json.dumps(
        result["segmentation"]["volumes"],
        indent=2
    ))

    print("\nInference Timings:")
    print(json.dumps(
        result["inference_timings"],
        indent=2
    ))

    print("\nTotal Request Time:",
          round(elapsed, 2), "seconds")

    # 6. Verify generated segmentation mask
    mask_path = Path(
        result["segmentation"]["mask_path"]
    )

    print("\nMask exists:", mask_path.exists())

    if mask_path.exists():
        mask_img = nib.load(str(mask_path))
        mask = np.asarray(mask_img.dataobj)

        print("Mask shape:", mask.shape)
        print("Unique labels:", np.unique(mask))
        print("\nPREDICTION TEST PASSED")


t1: 2.28 MB
t1ce: 2.29 MB
t2: 2.45 MB
flair: 2.25 MB

Backend is ready.

Sending MRI files to the AI model...
Radiomics: disabled for this test

HTTP Status: 500
API Error:
{"detail":"Segmentation failed: ModuleNotFoundError: No module named 'bmts'"}


In [34]:
import sys
import importlib.util
import subprocess

# 1. Python used by Colab notebook
print("NOTEBOOK PYTHON:")
print(sys.executable)

# 2. Check nnU-Net availability
print("\nNNU-NET CHECK:")

spec = importlib.util.find_spec("nnunetv2")

if spec:
    print("nnunetv2: FOUND")
    print("Module location:", spec.origin)
else:
    print("nnunetv2: NOT FOUND")

# 3. Check installed package
print("\nPIP PACKAGE CHECK:")

result = subprocess.run(
    [sys.executable, "-m", "pip", "show", "nnunetv2"],
    capture_output=True,
    text=True
)

print(result.stdout if result.stdout else result.stderr)

# 4. Inspect running backend process
print("\nRUNNING API PROCESSES:")

processes = subprocess.run(
    [
        "bash",
        "-lc",
        "ps -eo pid,ppid,args | grep -E '[u]vicorn|[m]ain:app|[a]pi_backend'"
    ],
    capture_output=True,
    text=True
)

print(processes.stdout)

NOTEBOOK PYTHON:
/usr/bin/python3

NNU-NET CHECK:
nnunetv2: NOT FOUND

PIP PACKAGE CHECK:


RUNNING API PROCESSES:
   3132    1434 /usr/bin/python3 -m uvicorn main:app --host 0.0.0.0 --port 8000 --log-level info



In [39]:
import sys
import subprocess
import importlib.util

print("Installing nnU-Net in:", sys.executable)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "nnunetv2==2.8.1"
])

# Verify installation
spec = importlib.util.find_spec("nnunetv2")

if spec is None:
    raise RuntimeError("nnunetv2 installation failed!")

import nnunetv2

print("\nNNU-NET INSTALLED SUCCESSFULLY!")
print("Module location:", spec.origin)
print("Python:", sys.executable)

Installing nnU-Net in: /usr/bin/python3

NNU-NET INSTALLED SUCCESSFULLY!
Module location: /usr/local/lib/python3.13/dist-packages/nnunetv2/__init__.py
Python: /usr/bin/python3


In [40]:

import os
import sys
import time
import signal
import subprocess
import requests

from pathlib import Path

# 1. Verify nnU-Net
import nnunetv2

print("nnU-Net:", "READY")
print("Python:", sys.executable)

# 2. Identify the existing API process
old_pid = 3132
proc_dir = Path(f"/proc/{old_pid}")

assert proc_dir.exists(), "Old API process was not found."

cmdline = (
    (proc_dir / "cmdline")
    .read_bytes()
    .decode(errors="replace")
    .replace("\x00", " ")
)

assert "uvicorn" in cmdline and "main:app" in cmdline, (
    "PID does not match the expected API process."
)

api_cwd = Path(os.readlink(proc_dir / "cwd"))

print("Backend directory:", api_cwd)

assert (api_cwd / "main.py").exists(), (
    "main.py was not found in backend directory."
)

# 3. Stop old API process only
os.kill(old_pid, signal.SIGTERM)

for _ in range(15):
    stat_file = proc_dir / "stat"

    if not stat_file.exists():
        break

    process_state = stat_file.read_text().split()[2]

    if process_state == "Z":
        break

    time.sleep(1)
else:
    os.kill(old_pid, signal.SIGKILL)
    time.sleep(2)

print("Old API process stopped.")

# 4. Start API using the current Python environment
api_log = open("/content/brainmri_api.log", "w")

api_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "main:app",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--log-level", "info",
    ],
    cwd=str(api_cwd),
    stdout=api_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

# 5. Wait for API health
api_ready = False

for _ in range(30):
    try:
        response = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=3,
        )

        if response.status_code == 200:
            api_ready = True
            break

    except requests.RequestException:
        pass

    time.sleep(1)

print("\nNew API PID:", api_process.pid)
print("Backend READY:", api_ready)

if api_ready:
    print("Health response:")
    print(response.json())
else:
    print("Backend startup failed.")
    print(
        Path("/content/brainmri_api.log")
        .read_text(errors="replace")[-7000:]
    )


nnU-Net: READY
Python: /usr/bin/python3


AssertionError: Old API process was not found.

In [41]:

import requests
import json
import time
import nibabel as nib
import numpy as np

from pathlib import Path

API_URL = "http://127.0.0.1:8000"

CASE_DIR = Path(
    "/content/drive/MyDrive/BrainMRI_Data/processed/"
    "brats2021/BraTS2021_00110/0"
)

modalities = ["t1", "t1ce", "t2", "flair"]

paths = {
    name: CASE_DIR / f"{name}.nii.gz"
    for name in modalities
}

# Check input files
for name, path in paths.items():
    assert path.exists(), f"Missing file: {path}"

    print(
        f"{name}: "
        f"{path.stat().st_size / (1024**2):.2f} MB"
    )

# Check backend
health = requests.get(
    f"{API_URL}/health",
    timeout=15
)
health.raise_for_status()

print("\nBackend is ready.")
print("Sending MRI files to the model...")

file_handles = {}

try:
    for name, path in paths.items():
        file_handles[name] = (
            path.name,
            open(path, "rb"),
            "application/octet-stream",
        )

    start_time = time.time()

    response = requests.post(
        f"{API_URL}/predict",
        files=file_handles,
        data={
            "case_id": "BraTS2021_00110",
            "run_radiomics": "false",
        },
        timeout=(30, 1800),
    )

    elapsed = time.time() - start_time

finally:
    for _, handle, _ in file_handles.values():
        handle.close()

print("\nHTTP Status:", response.status_code)
print("Request time:", round(elapsed, 2), "seconds")

if response.status_code != 200:
    print("\nAPI ERROR:")
    print(response.text[:5000])

else:
    data = response.json()

    print("\nFULL API RESPONSE:")
    print(json.dumps(data, indent=2))

    result = data.get("result", {})
    segmentation = result.get("segmentation", {})

    print("\nSegmentation status:")
    print(segmentation.get("status"))

    print("\nTumor measurements:")
    print(json.dumps(
        segmentation.get("volumes", {}),
        indent=2
    ))

    # Verify generated mask
    mask_path = segmentation.get("mask_path")

    if mask_path:
        mask_path = Path(mask_path)

        print("\nMask exists:", mask_path.exists())

        if mask_path.exists():
            mask_img = nib.load(str(mask_path))
            mask = np.asarray(mask_img.dataobj)

            print("Mask shape:", mask.shape)
            print("Unique labels:", np.unique(mask))

            print("\nPREDICTION TEST PASSED")

        else:
            print("Mask path was returned but file was not found.")

    else:
        print("No mask_path found in the API response.")


t1: 2.28 MB
t1ce: 2.29 MB
t2: 2.45 MB
flair: 2.25 MB

Backend is ready.
Sending MRI files to the model...

HTTP Status: 500
Request time: 0.31 seconds

API ERROR:
{"detail":"Segmentation failed: ModuleNotFoundError: No module named 'bmts'"}


In [51]:

import os
import sys
import time
import signal
import subprocess
import requests

from pathlib import Path

# 1. Locate project and backend
REPO = Path("/content/repo")
REPO_SRC = REPO / "src"

API_DIR = Path(
    "/content/drive/.shortcut-targets-by-id/"
    "1YnS9k2ISGyxs4jmGKUgbH5-0Xnlm11VR/"
    "BrainMRI_Data/api_backend"
)

assert (REPO_SRC / "bmts" / "__init__.py").exists()
assert (API_DIR / "main.py").exists()

# 2. Prepare the correct Python environment
server_env = os.environ.copy()

server_env["PYTHONPATH"] = (
    str(REPO_SRC)
    + os.pathsep
    + server_env.get("PYTHONPATH", "")
)

server_env["PYTHONUNBUFFERED"] = "1"

# 3. Test imports BEFORE stopping the current server
test = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import bmts; "
            "from bmts.deploy.segmenter import FastSegmenter; "
            "from nnunetv2.inference.predict_from_raw_data "
            "import nnUNetPredictor; "
            "print('ALL MODEL IMPORTS OK')"
        ),
    ],
    cwd=str(API_DIR),
    env=server_env,
    capture_output=True,
    text=True,
)

print("Import test:")
print(test.stdout)
print(test.stderr[-2000:] if test.stderr else "")

if test.returncode != 0:
    raise RuntimeError(
        "Import test failed. The existing API was not stopped."
    )

# 4. Stop old API process
old_pid = api_process.pid

cmdline = Path(
    f"/proc/{old_pid}/cmdline"
).read_bytes().decode(errors="replace")

assert "uvicorn" in cmdline and "main:app" in cmdline

os.kill(old_pid, signal.SIGTERM)

try:
    api_process.wait(timeout=15)
except subprocess.TimeoutExpired:
    api_process.kill()
    api_process.wait()

print("Old API stopped.")

# 5. Start API with PYTHONPATH configured
server_log_handle = open(
    "/content/brainmri_api.log",
    "w",
    buffering=1,
)

api_process = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "main:app",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--log-level", "info",
    ],
    cwd=str(API_DIR),
    env=server_env,
    stdout=server_log_handle,
    stderr=subprocess.STDOUT,
)

# 6. Wait for health check
ready = False

for _ in range(40):
    if api_process.poll() is not None:
        break

    try:
        response = requests.get(
            "http://127.0.0.1:8000/health",
            timeout=3,
        )

        if response.status_code == 200:
            ready = True
            break

    except requests.RequestException:
        pass

    time.sleep(1)

print("\nNew API PID:", api_process.pid)
print("Backend READY:", ready)

if ready:
    print("Health:", response.json())
else:
    print(
        Path("/content/brainmri_api.log")
        .read_text(errors="replace")[-7000:]
    )


Import test:
ALL MODEL IMPORTS OK


Old API stopped.

New API PID: 56232
Backend READY: True
Health: {'status': 'ok', 'service': 'Brain MRI Segmentation API', 'export_dir_exists': True, 'export_manifest_exists': True, 'device': 'cpu', 'tta_enabled': False, 'supported_file_extensions': ['.nii', '.nii.gz'], 'modalities_required': ['t1', 't1ce', 't2', 'flair']}


In [52]:

import requests
import json
import time
import nibabel as nib
import numpy as np

from pathlib import Path

API_URL = "http://127.0.0.1:8000"

CASE_DIR = Path(
    "/content/drive/MyDrive/BrainMRI_Data/processed/"
    "brats2021/BraTS2021_00110/0"
)

modalities = ["t1", "t1ce", "t2", "flair"]

paths = {
    name: CASE_DIR / f"{name}.nii.gz"
    for name in modalities
}

# Verify input files
for name, path in paths.items():
    assert path.exists(), f"Missing file: {path}"
    print(f"{name}: {path.stat().st_size / (1024**2):.2f} MB")

# Verify backend
health = requests.get(
    f"{API_URL}/health",
    timeout=15
)
health.raise_for_status()

print("\nBackend is ready.")
print("Sending MRI files to the model...")

file_handles = {}

try:
    for name, path in paths.items():
        file_handles[name] = (
            path.name,
            open(path, "rb"),
            "application/octet-stream",
        )

    start_time = time.time()

    response = requests.post(
        f"{API_URL}/predict",
        files=file_handles,
        data={
            "case_id": "BraTS2021_00110",
            "run_radiomics": "false",
        },
        timeout=(30, 1800),
    )

    elapsed = time.time() - start_time

finally:
    for _, handle, _ in file_handles.values():
        handle.close()

print("\nHTTP Status:", response.status_code)
print("Request time:", round(elapsed, 2), "seconds")

if response.status_code != 200:
    print("\nAPI ERROR:")
    print(response.text[:5000])

else:
    data = response.json()
    result = data.get("result", {})
    segmentation = result.get("segmentation", {})

    print("\nSegmentation Status:")
    print(segmentation.get("status"))

    print("\nTumor Measurements:")
    print(json.dumps(
        segmentation.get("volumes", {}),
        indent=2
    ))

    print("\nInference Timings:")
    print(json.dumps(
        result.get("inference_timings", {}),
        indent=2
    ))

    mask_path = segmentation.get("mask_path")

    if mask_path:
        mask_path = Path(mask_path)

        print("\nMask exists:", mask_path.exists())

        if mask_path.exists():
            mask_img = nib.load(str(mask_path))
            mask = np.asarray(mask_img.dataobj)

            print("Mask shape:", mask.shape)
            print("Unique labels:", np.unique(mask))
            print("\nPREDICTION TEST PASSED")
    else:
        print("\nNo mask_path returned by API.")
        print(json.dumps(data, indent=2))


t1: 2.28 MB
t1ce: 2.29 MB
t2: 2.45 MB
flair: 2.25 MB

Backend is ready.
Sending MRI files to the model...

HTTP Status: 200
Request time: 114.96 seconds

Segmentation Status:
success

Tumor Measurements:
{
  "TC": {
    "voxels": 14671,
    "volume_cm3": 14.671
  },
  "WT": {
    "voxels": 90977,
    "volume_cm3": 90.977
  },
  "ET": {
    "voxels": 14638,
    "volume_cm3": 14.638
  },
  "label_counts": {
    "0": 8837023,
    "1": 33,
    "2": 76306,
    "3": 14638
  }
}

Inference Timings:
{
  "preprocess_s": 1.9743466780000745,
  "network_s": 107.49917034200007,
  "export_s": 0.26523516299857874,
  "total_s": 109.73875218299872
}

Mask exists: True
Mask shape: (240, 240, 155)
Unique labels: [0. 1. 2. 3.]

PREDICTION TEST PASSED


In [53]:
from google.colab import files
from pathlib import Path
import py_compile

uploaded = files.upload()

if len(uploaded) != 1:
    raise ValueError("Upload exactly one Python file.")

filename, content = next(iter(uploaded.items()))

target = Path("/content/repo/apps/web_interface.py")
target.write_bytes(content)

# Check syntax
py_compile.compile(
    str(target),
    doraise=True
)

print("New frontend uploaded successfully!")
print("Location:", target)
print("Syntax: OK")

Saving web_interface_m4_t03.py to web_interface_m4_t03 (1).py
New frontend uploaded successfully!
Location: /content/repo/apps/web_interface.py
Syntax: OK


In [54]:
from google.colab import output

output.serve_kernel_port_as_iframe(
    8501,
    path="/",
    height=900
)

<IPython.core.display.Javascript object>

In [55]:
from pathlib import Path

processes = {
    "FastAPI Backend": api_process.pid,
    "Streamlit Frontend": streamlit_process.pid,
}

for name, pid in processes.items():
    status_file = Path(f"/proc/{pid}/status")

    print(f"\n{name} (PID: {pid})")

    if not status_file.exists():
        print("Process not found")
        continue

    status = status_file.read_text()

    for line in status.splitlines():
        if line.startswith(("VmRSS:", "VmHWM:")):
            print(line)


FastAPI Backend (PID: 56232)
VmHWM:	 2386520 kB
VmRSS:	  747576 kB

Streamlit Frontend (PID: 19430)
VmHWM:	 1234156 kB
VmRSS:	 1100332 kB


In [56]:

import requests
import subprocess
import time
import re
from pathlib import Path

# 1. Check both services
frontend = requests.get(
    "http://127.0.0.1:8501/_stcore/health",
    timeout=10
)

backend = requests.get(
    "http://127.0.0.1:8000/health",
    timeout=10
)

assert frontend.status_code == 200, "Streamlit is not running"
assert backend.status_code == 200, "FastAPI is not running"

print("Streamlit: READY")
print("FastAPI: READY")

# 2. Install Cloudflare Tunnel
binary = Path("/usr/local/bin/cloudflared")

if not binary.exists():
    subprocess.run(
        [
            "bash", "-lc",
            "curl -L --fail --silent --show-error "
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
            "-o /usr/local/bin/cloudflared && "
            "chmod +x /usr/local/bin/cloudflared"
        ],
        check=True
    )

print("Cloudflared:", subprocess.check_output(
    [str(binary), "--version"], text=True
).strip())

# 3. Start public tunnel to Streamlit
log_path = Path("/content/cloudflared.log")
cloudflared_log = open(log_path, "w")

cloudflared_process = subprocess.Popen(
    [
        str(binary),
        "tunnel",
        "--no-autoupdate",
        "--url",
        "http://127.0.0.1:8501",
    ],
    stdout=cloudflared_log,
    stderr=subprocess.STDOUT,
    start_new_session=True,
)

# 4. Extract the public URL
public_url = None

for _ in range(60):
    if cloudflared_process.poll() is not None:
        break

    logs = log_path.read_text(errors="replace")

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        logs
    )

    if match:
        public_url = match.group(0)
        break

    time.sleep(1)

if public_url:
    print("\n" + "=" * 50)
    print("YOUR BRAIN MRI APP IS LIVE!")
    print("=" * 50)
    print("\nSHARE THIS LINK WITH YOUR FRIENDS:")
    print(public_url)
    print("\nKeep Colab and all processes running.")
else:
    print("Tunnel URL not found.")
    print(log_path.read_text(errors="replace")[-5000:])


Streamlit: READY
FastAPI: READY
Cloudflared: cloudflared version 2026.10.7 (built 2026-10-30-17:05 UTC)

YOUR BRAIN MRI APP IS LIVE!

SHARE THIS LINK WITH YOUR FRIENDS:
https://materials-finds-optimize-gras.trycloudflare.com

Keep Colab and all processes running.


In [57]:

import requests
import time
from pathlib import Path

PUBLIC_URL = (
    "https://translator-receivers-function-process.trycloudflare.com"
)

# 1. Check Cloudflare process
print("========== TUNNEL PROCESS ==========")

try:
    print("Process status:", cloudflared_process.poll())
    print("None means the tunnel is still running.")
except NameError:
    print("Cloudflared process variable not found.")

# 2. Check Streamlit locally
print("\n========== LOCAL STREAMLIT ==========")

try:
    r = requests.get(
        "http://127.0.0.1:8501/_stcore/health",
        timeout=10
    )
    print("Local status:", r.status_code)
except Exception as e:
    print("Local error:", str(e))

# 3. Check tunnel logs
print("\n========== CLOUDFLARE LOGS ==========")

log_path = Path("/content/cloudflared.log")

if log_path.exists():
    logs = log_path.read_text(errors="replace")
    print(logs[-6000:])
else:
    print("Log file not found.")

# 4. Test public URL from Colab
print("\n========== PUBLIC URL TEST ==========")

try:
    r = requests.get(
        PUBLIC_URL + "/_stcore/health",
        timeout=25
    )

    print("Public status:", r.status_code)
    print("Server:", r.headers.get("server"))
    print("Response:", r.text[:1000])

except Exception as e:
    print("Public connection error:", repr(e))


========== TUNNEL PROCESS ==========
Process status: None
None means the tunnel is still running.

========== LOCAL STREAMLIT ==========
Local status: 200

========== CLOUDFLARE LOGS ==========
2026-11-02T12:08:59Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-02T12:08:59Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-11-02T12:09:02Z INF +--------------------------------------------------------------------------------------------+
2026-11

In [58]:
import requests

PUBLIC_URL = (
    "https://translator-receivers-function-process.trycloudflare.com"
)

r = requests.get(
    PUBLIC_URL + "/",
    timeout=20
)

print("Homepage Status:", r.status_code)
print("Content Type:", r.headers.get("content-type"))
print("HTML Size:", len(r.text))
print("Streamlit HTML:", "streamlit" in r.text.lower())

Homepage Status: 200
Content Type: text/html; charset=utf-8
HTML Size: 6467
Streamlit HTML: True


In [59]:

# ============================================================
# BRAIN MRI APP - ONE CELL LAUNCHER
# Starts Drive + API + Streamlit + Cloudflare Tunnel
# ============================================================

import os
import sys
import re
import time
import shutil
import subprocess
import importlib.util
from pathlib import Path

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

from google.colab import drive

if not os.path.ismount("/content/drive"):
    print("Connecting Google Drive...")
    drive.mount("/content/drive")

BASE = Path("/content/drive/MyDrive/BrainMRI_Data")
REPO = Path("/content/repo")

API_DIR = BASE / "api_backend"
EXPORT_DIR = BASE / "models/seg-model-v1-torchscript"
OUTPUT_DIR = BASE / "runs/API_Backend"

APP_BACKUP = BASE / "deployment/web_interface.py"
APP_PATH = REPO / "apps/web_interface.py"

assert BASE.is_dir(), f"Project folder not found: {BASE}"
assert API_DIR.joinpath("main.py").is_file(), "FastAPI files not found."
assert EXPORT_DIR.joinpath("deploy_manifest.json").is_file(), "Model export not found."

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Google Drive: READY")
print("Model export: FOUND")
print("Backend files: FOUND")


# ------------------------------------------------------------
# 2. Get tested repository version
# ------------------------------------------------------------

if not (REPO / ".git").exists():
    print("\nCloning project repository...")

    subprocess.run(
        [
            "git", "clone", "-q",
            "https://github.com/nhahub/NHA-5-187.git",
            str(REPO)
        ],
        check=True
    )

subprocess.run(
    ["git", "-C", str(REPO), "checkout", "-q", "2e0e67d"],
    check=True
)

assert (REPO / "src/bmts").is_dir(), "BMTS package missing."

print("Repository: READY")


# ------------------------------------------------------------
# 3. Restore your web interface from Google Drive
# ------------------------------------------------------------

APP_PATH.parent.mkdir(parents=True, exist_ok=True)

if APP_BACKUP.is_file():

    shutil.copy2(APP_BACKUP, APP_PATH)
    print("Frontend restored from Google Drive.")

elif APP_PATH.is_file() and "M4-T03" in APP_PATH.read_text(
    encoding="utf-8", errors="ignore"
):

    print("Existing frontend found; keeping it.")

else:

    from google.colab import files

    print("Upload your web_interface.py file:")
    uploaded = files.upload()

    if len(uploaded) != 1:
        raise RuntimeError("Upload exactly one web_interface.py file.")

    filename, content = next(iter(uploaded.items()))

    if not filename.endswith("web_interface.py"):
        raise ValueError("Please upload web_interface.py")

    APP_PATH.write_bytes(content)

    APP_BACKUP.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(APP_PATH, APP_BACKUP)

    print("Frontend uploaded and backed up to Drive.")

assert APP_PATH.is_file()


# ------------------------------------------------------------
# 4. Install missing Python packages only
# ------------------------------------------------------------

packages = {
    "torch": "torch",
    "nnunetv2": "nnunetv2==2.8.1",
    "fastapi": "fastapi>=0.115,<1.0",
    "uvicorn": "uvicorn[standard]>=0.30,<1.0",
    "multipart": "python-multipart",
    "nibabel": "nibabel>=5.2",
    "SimpleITK": "SimpleITK>=2.2.1",
    "pandas": "pandas>=2.0",
    "requests": "requests",
    "streamlit": "streamlit==1.66.1",
    "matplotlib": "matplotlib>=3.8",
}

missing = [
    package
    for module, package in packages.items()
    if importlib.util.find_spec(module) is None
]

if missing:
    print("\nInstalling missing packages:")
    print(missing)

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            *packages.values(),
        ],
        check=True
    )
else:
    print("\nPython dependencies: READY")


# ------------------------------------------------------------
# 5. Configure model and backend environment
# ------------------------------------------------------------

import requests

WORK_DIR = Path("/content/nnunet_work")
(WORK_DIR / "nnUNet_raw").mkdir(parents=True, exist_ok=True)
(WORK_DIR / "nnUNet_preprocessed").mkdir(parents=True, exist_ok=True)

server_env = os.environ.copy()

server_env.update({
    "BRAINMRI_BASE": str(BASE),
    "BRAINMRI_EXPORT_DIR": str(EXPORT_DIR),
    "BRAINMRI_API_OUTPUT": str(OUTPUT_DIR),
    "BRAINMRI_DEVICE": "cpu",
    "BRAINMRI_API_URL": "http://127.0.0.1:8000",
    "API_TTA": "false",
    "PYTHONUNBUFFERED": "1",
    "nnUNet_raw": str(WORK_DIR / "nnUNet_raw"),
    "nnUNet_preprocessed": str(WORK_DIR / "nnUNet_preprocessed"),
    "nnUNet_results": str(BASE / "runs/M2-T04_nnunet/nnUNet_results"),
    "PYTHONPATH": (
        str(REPO) + os.pathsep +
        str(REPO / "src") + os.pathsep +
        os.environ.get("PYTHONPATH", "")
    ),
})

# Verify all important model imports before starting
probe = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import torch, nnunetv2, bmts; "
            "from bmts.deploy.segmenter import FastSegmenter; "
            "print('MODEL IMPORTS OK')"
        ),
    ],
    cwd=str(API_DIR),
    env=server_env,
    capture_output=True,
    text=True
)

if probe.returncode != 0:
    print(probe.stderr[-5000:])
    raise RuntimeError("Model imports failed.")

print(probe.stdout.strip())


# ------------------------------------------------------------
# 6. Start FastAPI if it is not already running
# ------------------------------------------------------------

def service_ok(url):
    try:
        response = requests.get(url, timeout=5)
        return response.status_code == 200
    except requests.RequestException:
        return False


if service_ok("http://127.0.0.1:8000/health"):

    print("FastAPI: ALREADY RUNNING")

else:

    print("Starting FastAPI...")

    api_log_handle = open(
        "/content/brainmri_api.log",
        "w",
        buffering=1
    )

    api_process = subprocess.Popen(
        [
            sys.executable,
            "-m", "uvicorn",
            "main:app",
            "--host", "0.0.0.0",
            "--port", "8000",
            "--log-level", "info",
        ],
        cwd=str(API_DIR),
        env=server_env,
        stdout=api_log_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

    for _ in range(60):
        if api_process.poll() is not None:
            break

        if service_ok("http://127.0.0.1:8000/health"):
            break

        time.sleep(1)

    if not service_ok("http://127.0.0.1:8000/health"):
        print(
            Path("/content/brainmri_api.log")
            .read_text(errors="replace")[-6000:]
        )
        raise RuntimeError("FastAPI failed to start.")

print("FastAPI: READY")


# ------------------------------------------------------------
# 7. Start Streamlit if it is not already running
# ------------------------------------------------------------

if service_ok("http://127.0.0.1:8501/_stcore/health"):

    print("Streamlit: ALREADY RUNNING")

else:

    print("Starting Streamlit...")

    streamlit_log_handle = open(
        "/content/streamlit.log",
        "w",
        buffering=1
    )

    streamlit_process = subprocess.Popen(
        [
            sys.executable,
            "-m", "streamlit",
            "run", str(APP_PATH),
            "--server.port=8501",
            "--server.address=0.0.0.0",
            "--server.headless=true",
            "--server.enableCORS=false",
            "--server.enableXsrfProtection=false",
            "--server.allowedHosts=*",
            "--browser.gatherUsageStats=false",
        ],
        cwd=str(REPO),
        env=server_env,
        stdout=streamlit_log_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

    for _ in range(40):
        if streamlit_process.poll() is not None:
            break

        if service_ok("http://127.0.0.1:8501/_stcore/health"):
            break

        time.sleep(1)

    if not service_ok("http://127.0.0.1:8501/_stcore/health"):
        print(
            Path("/content/streamlit.log")
            .read_text(errors="replace")[-6000:]
        )
        raise RuntimeError("Streamlit failed to start.")

print("Streamlit: READY")


# ------------------------------------------------------------
# 8. Install Cloudflare Tunnel if needed
# ------------------------------------------------------------

CLOUDFLARED = Path("/usr/local/bin/cloudflared")

if not CLOUDFLARED.exists():

    print("Downloading Cloudflare Tunnel...")

    subprocess.run(
        [
            "bash", "-lc",
            "curl -L --fail --silent --show-error "
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 "
            "-o /usr/local/bin/cloudflared && "
            "chmod +x /usr/local/bin/cloudflared"
        ],
        check=True
    )

print("Cloudflared: READY")


# ------------------------------------------------------------
# 9. Start or reuse Cloudflare Tunnel
# ------------------------------------------------------------

TUNNEL_LOG = Path("/content/cloudflared.log")

tunnel_url = None

# Reuse an existing tunnel if this cell is run again
if (
    "cloudflared_process" in globals()
    and cloudflared_process.poll() is None
    and TUNNEL_LOG.exists()
):

    logs = TUNNEL_LOG.read_text(errors="replace")

    match = re.search(
        r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
        logs
    )

    if match:
        tunnel_url = match.group(0)
        print("Reusing existing Tunnel.")

# Otherwise create a new one
if not tunnel_url:

    TUNNEL_LOG.write_text("")

    cloudflared_log_handle = open(
        TUNNEL_LOG,
        "w",
        buffering=1
    )

    cloudflared_process = subprocess.Popen(
        [
            str(CLOUDFLARED),
            "tunnel",
            "--no-autoupdate",
            "--url",
            "http://127.0.0.1:8501",
        ],
        stdout=cloudflared_log_handle,
        stderr=subprocess.STDOUT,
        start_new_session=True
    )

    print("Starting public Tunnel...")

    for _ in range(60):

        if cloudflared_process.poll() is not None:
            break

        logs = TUNNEL_LOG.read_text(errors="replace")

        match = re.search(
            r"https://[a-zA-Z0-9-]+\.trycloudflare\.com",
            logs
        )

        if match:
            tunnel_url = match.group(0)
            break

        time.sleep(1)

if not tunnel_url:

    print(
        TUNNEL_LOG.read_text(errors="replace")[-6000:]
    )

    raise RuntimeError("Could not create Cloudflare Tunnel.")


# ------------------------------------------------------------
# 10. Final health checks and shareable link
# ------------------------------------------------------------

print("\nChecking public application...")

public_ok = False

for _ in range(20):

    try:
        public_response = requests.get(
            tunnel_url + "/_stcore/health",
            timeout=10
        )

        if public_response.status_code == 200:
            public_ok = True
            break

    except requests.RequestException:
        pass

    time.sleep(2)

print("\n" + "=" * 55)
print("       BRAIN MRI ANALYSIS - LAUNCH COMPLETE")
print("=" * 55)

print("FastAPI:       READY")
print("Streamlit:     READY")
print("Cloudflare:    RUNNING")
print("Public check:", public_response.status_code if public_ok else "Not verified")

print("\nYOUR SHAREABLE APP LINK:")
print(tunnel_url)

print("\nKeep this Colab runtime connected.")
print("Do not click Runtime > Disconnect or Factory reset.")
print("=" * 55)


Google Drive: READY
Model export: FOUND
Backend files: FOUND
Repository: READY
Existing frontend found; keeping it.

Python dependencies: READY
MODEL IMPORTS OK
FastAPI: ALREADY RUNNING
FastAPI: READY
Streamlit: ALREADY RUNNING
Streamlit: READY
Cloudflared: READY
Reusing existing Tunnel.

Checking public application...

       BRAIN MRI ANALYSIS - LAUNCH COMPLETE
FastAPI:       READY
Streamlit:     READY
Cloudflare:    RUNNING
Public check: 200

YOUR SHAREABLE APP LINK:
https://materials-finds-optimize-gras.trycloudflare.com

Keep this Colab runtime connected.
Do not click Runtime > Disconnect or Factory reset.


## Expected result

- API backend files are written to `BrainMRI_Data/api_backend/`.
- Run outputs are written to `BrainMRI_Data/runs/API_Backend/`.
- The model checkpoint is verified against `models/registry/seg-model/v1/manifest.json`.
- The exported TorchScript model is reused when valid or created by the repository export script when missing.
- Invalid extensions are rejected with HTTP `415 Unsupported Media Type`.\n- PyRadiomics is optional; if unavailable, segmentation still runs and the response reports radiomics as unavailable.

**Validation boundary:** this notebook performs syntax checks and API smoke tests. A full inference test requires real cleaned NIfTI files from the project and was not run during notebook generation.
